# 49. 선정 파이프라인 신규 질문·반복 확인

48번이 끝나고 선정된 설정을 고정한 뒤 실행합니다. 새32질문(명시카드/수치조건/의미혜택/부정진단 각8), 두 운영 프로필, 비구조화 기준 U와 선정 순차 검색 S를 비교합니다. 최대3회 신규 답변, 384논리조건입니다. 질문과 정답 주석은 분리하며 문서 자체의 holdout은 아닙니다.

1회차를 먼저 완료하고 방식명을 가린 원문 검토를 기다립니다. 새 중요오류가 발견된 프로필은 나머지2회 유료 호출을 중단합니다. 중단/미실행을 오답이나 동률로 채우지 않습니다. 새 반복은 별도 request identity를 쓰고 동일 질문/근거라도 답변을 재사용하지 않습니다. 임베딩과 검색 점수 캐시는 재사용할 수 있습니다. 전체8달러 상한, 운영 코드 변경 없음. 공통 검색/API 기록만 기존47보조 코드를 사용하고 HTML은 직접 작성합니다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,hashlib,time,os,copy,ast
import nbformat
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
def load(name,filename):
 spec=importlib.util.spec_from_file_location(name,ROOT/'notebooks/scripts'/filename)
 module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module);return module
R=load('stage49_runtime','47_retrieval_runtime.py');P=R.P
PATH=P.F.STAGES[49];PATH.mkdir(exist_ok=True);P.F.initialize(49)
prior=P.F.STAGES[48]
assert P.F.read_json(prior/'completion_status.json')['status']=='complete'
selected=P.F.read_json(prior/'selected_policy.json')
assert P.F.read_json(P.F.STAGES[47]/'selected_policy.json')['partial_prioritization'] is False
queries=P.F.read_json(PATH/'question_draft_before_48_selection.json')['queries']
annotations=P.F.read_json(PATH/'annotation_draft_before_48_selection.json')
assert len(queries)==32 and len({q['query_id'] for q in queries})==32
source_audit=P.F.read_json(PATH/'prepaid_source_audit.json');assert source_audit['passed']
assert source_audit['questions_sha256']==P.sha((PATH/'question_draft_before_48_selection.json').read_bytes())
assert source_audit['annotation_sha256']==P.sha((PATH/'annotation_draft_before_48_selection.json').read_bytes())
assert all(P.sha((ROOT/path).read_bytes())==value for path,value in source_audit['source_file_hashes'].items())
catalog=P.F.read_json(P.DATA/'42_expanded_query_evaluation/catalog.json')['catalog']
facts=[json.loads(s) for s in (P.DATA/'30_structured_benefit_db/facts.jsonl').read_text().splitlines()]
fact_by_id={f['fact_id']:f for f in facts}
mapping=P.F.read_json(P.DATA/'45_evidence_supplement_comparison/prepared_packages.json')['mapping']
# 48번에서 동결한 보조 함수만 재사용합니다. 노트북의 실행/과거 파일 쓰기는 호출하지 않습니다.
source48=next(c.source for c in nbformat.read(ROOT/'notebooks/48_structured_retrieval_guidance.ipynb',4).cells if c.metadata.get('tags')==['guidance_contract'])
tree=ast.parse(source48)
for node in tree.body:
 if isinstance(node,ast.FunctionDef) and node.name=='guidance' or isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='FIELD_LABELS' for t in node.targets):
  exec(compile(ast.Module(body=[node],type_ignores=[]),'<frozen48-guidance>','exec'),globals())
PROMPT=load('stage49_prompt','33_query_routing.py').INSTRUCTIONS+'\n카드 catalog:\n'+P.F.canonical(catalog)
profiles=['card_page_section_benefit','parent_child_bundle'];modes=['U','S']
ALLOW_NETWORK=os.getenv('PREPROD49_ALLOW_NETWORK')=='1'
def write(name,value):P.F.write_json(PATH/name,value)
def frozen(name,value):
 path=PATH/name
 if path.exists():assert P.F.read_json(path)==value,'봉인 변경: '+name
 else:write(name,value)
def assert_notebook_hash(path,expected):
 actual=P.sha([c.source for c in nbformat.read(path,4).cells if c.cell_type=='code'])
 if actual!=expected:raise RuntimeError('유료 실행 노트북 코드가 봉인 뒤 변경되었습니다.')
def guard_current_notebook(manifest):
 assert_notebook_hash(ROOT/'notebooks/49_selected_pipeline_confirmation.ipynb',manifest['notebook_code_sha256'])
class ContractCache(R.BoundCache):
 def reserve(self,normalized):
  guard_current_notebook(self.manifest)
  return super().reserve(normalized)
class ContractRuntime(R.Runtime):
 def embeddings(self,texts,*,allow_network=False):
  batches=[]
  for offset in range(0,len(texts),32):
   guard_current_notebook(self.manifest)
   batches.extend(super().embeddings(texts[offset:offset+32],allow_network=allow_network))
  return self.np.asarray(batches,dtype=self.np.float32)
 def search(self,*args,**kwargs):
  guard_current_notebook(self.manifest)
  return super().search(*args,**kwargs)
def answer_schema_hash(evidence):
 if not evidence:return None
 from pickcardu_rag.answering import _generated_answer_model
 return P.sha(_generated_answer_model(len(evidence)).model_json_schema())
def prefix_contract(contract,repeat):
 result=copy.deepcopy(contract)
 for key in ['input_sha256_by_identity','schema_sha256_by_identity']:
  result[key]={identity:value for identity,value in contract[key].items() if int(identity.rsplit(':r',1)[1])<=repeat}
 return result
def selected_route(q,value,failed=False):
 trusted=P.explicit_scope(q,catalog)
 unclear=('스카이패스카드' in P.compact(q) and trusted is None)
 return R.selected_route(q,value,facts,catalog,trusted_scope=trusted,parser_failed=failed,scope_unclear=unclear)
print('48번 선정 확인. 신규 질문32, 후보와 기준의 카드 명시 범위는 동일하게 보존.')

48번 선정 확인. 신규 질문32, 후보와 기준의 카드 명시 범위는 동일하게 보존.


In [2]:
old=P.F.read_json(prior/'execution_manifest.json')
source_paths=[ROOT/p for p in old['source_sha256']]+[ROOT/'notebooks/48_structured_retrieval_guidance.ipynb']
data_paths=[ROOT/p for p in old['data_sha256']]+[prior/'selected_policy.json',prior/'completion_status.json',
 PATH/'question_draft_before_48_selection.json',PATH/'annotation_draft_before_48_selection.json',PATH/'prepaid_source_audit.json',
 *[ROOT/path for path in source_audit['source_file_hashes']]]
manifest=dict(stage=49,split='confirmatory_questions_same_corpus',model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,parser_output=3000,answer_output=2400,
 questions_sha256=P.sha(queries),annotations_sha256=P.sha(annotations),selected48=selected,
 notebook_code_sha256=P.sha([c.source for c in nbformat.read(ROOT/'notebooks/49_selected_pipeline_confirmation.ipynb',4).cells if c.cell_type=='code']),
 question_count=32,repeats=3,profiles=profiles,modes=modes,logical_answer_conditions=384,
 allowed_identities={'parser':[f"{q['query_id']}:parser:r{rep}" for rep in range(1,4) for q in queries],
 'answer':[f"{q['query_id']}:{pf}:{mode}:r{rep}" for rep in range(1,4) for q in queries for pf in profiles for mode in modes]},
 allowed_request_by_identity={f"{q['query_id']}:parser:r{rep}":P.sha([dict(role='user',content=q['question'])]) for rep in range(1,4) for q in queries},
 parser_instructions_sha256=P.sha(PROMPT),parser_schema_sha256=P.sha(P.E.QueryPlan.model_json_schema()),
 allowed_embedding_text_sha256=[P.sha(q['question']) for q in queries],
 source_sha256={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in source_paths},
 data_sha256={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in data_paths},
 maximum_answer_requests=768,maximum_parser_requests=96,maximum_embedding_requests=7,maximum_physical_requests=871,
 maximum_validation_retries_per_answer=1,maximum_network_retries=0,global_cost_cap_usd='8',stage_cost_cap_usd='4.5',
 parameters=dict(component_depth=50,candidates=20,top_cards=3,evidence_per_card=5,payload_bytes=64000,
 vector_weight=.4,parent_bundle='budget5',question_reranker=True),
 primary_unit='32 questions per profile, 0..3 PASS counts per method; not96 independent questions',
 pass_gate='mandatory9; no new important error/core development regression; zero question losses in every subgroup; >=1 question win',
 early_stop='after R1 source review, stop remaining2 paid repeats of affected profile on new important error; paired baseline repeats also omitted as unnecessary',
 fresh_answer_each_repeat=True,reuse_answer_across_modes=False,reuse_embeddings_and_retrieval_only=True,
 latency_sla='unset, measured only',operating_changes=False,new_document_embeddings=0)
assert manifest['max_output_tokens']==manifest['parser_output']==3000
assert manifest['maximum_answer_requests']==manifest['logical_answer_conditions']*2
assert manifest['maximum_physical_requests']==sum(manifest[k] for k in ['maximum_answer_requests','maximum_parser_requests','maximum_embedding_requests'])
frozen('execution_manifest.json',manifest);guard_current_notebook(manifest);frozen('questions.json',dict(queries=queries,split=manifest['split']))
frozen('evaluation_annotations.json',annotations)
# 46번 리터럴 필수 회귀9건을 현재 공유 정책으로 다시 실행합니다.
source46=next(c.source for c in nbformat.read(ROOT/'notebooks/46_query_boundary_validation.ipynb',4).cells if c.cell_type=='code' and 'REGRESSIONS =' in c.source)
fixture=P.F.read_json(P.DATA/'43_offline_validation/routing_fixtures.json');reg_ns={}
for node in ast.parse(source46).body:
 if isinstance(node,ast.FunctionDef) and node.name in ['req','plan'] or isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REGRESSIONS' for t in node.targets):
  exec(compile(ast.Module(body=[node],type_ignores=[]),'<literal-boundary-regression>','exec'),reg_ns)
regressions=[]
for c in reg_ns['REGRESSIONS']:
 got=R.selected_route('',c['value'],fixture['facts'],fixture['catalog'],trusted_scope=c['explicit'],parser_failed=c['failed'],scope_unclear=c['unclear'])
 regressions.append(dict(id=c['id'],passed=all(got[k]==v for k,v in c['want'].items())))
assert len(regressions)==9 and all(r['passed'] for r in regressions)
assert answer_schema_hash([]) is None
assert answer_schema_hash([{}]) is not None
write('mandatory_regression.json',dict(records=regressions,new_api_requests=0))
assert len({P.sha(dict(request='same',identity='r'+str(r))) for r in range(1,4)})==3
sample=dict(input_sha256_by_identity={'q:r1':'A'},schema_sha256_by_identity={'q:r1':'S'})
old_prefix=prefix_contract(sample,1)
sample['input_sha256_by_identity']['q:r2']='B';sample['schema_sha256_by_identity']['q:r2']='T'
assert prefix_contract(sample,1)==old_prefix
if not (PATH/'protected_files.json').exists():
 protected={r['path']:P.F.file_record(ROOT/r['path']) for r in P.F.read_json(prior/'protected_files.json')['files']}
 for source in prior.rglob('*'):
  if source.is_file() and not any(x in source.parts for x in ['indexes','execution_cache','contract_tests']):protected[str(source.relative_to(ROOT))]=P.F.file_record(source)
 for source in [ROOT/'notebooks/48_structured_retrieval_guidance.ipynb',ROOT/'docs/report/48_structured_retrieval_guidance_report.html']:
  protected[str(source.relative_to(ROOT))]=P.F.file_record(source)
 write('protected_files.json',dict(files=sorted(protected.values(),key=lambda r:r['path'])))
import tempfile
with tempfile.TemporaryDirectory(dir=PATH) as tmp:
 check_path=Path(tmp)/'fixture.ipynb';fixture_nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell('value=1')]);nbformat.write(fixture_nb,check_path)
 expected=P.sha(['value=1']);assert_notebook_hash(check_path,expected)
 fixture_nb.cells[0].execution_count=1;nbformat.write(fixture_nb,check_path);assert_notebook_hash(check_path,expected)
 fixture_nb.cells[0].source='value=2';nbformat.write(fixture_nb,check_path)
 try:assert_notebook_hash(check_path,expected)
 except RuntimeError:pass
 else:raise AssertionError('노트북 코드 변경을 거절하지 않음')
print('필수 경계9/9,384논리조건·최대871물리요청·전체8달러 봉인. 이 셀까지 API0회.')

필수 경계9/9,384논리조건·최대871물리요청·전체8달러 봉인. 이 셀까지 API0회.


In [3]:
from pickcardu_rag.answering import OpenAIService,serialize_answer_payload,_generated_answer_model
runtime=ContractRuntime(49)
question_vectors=runtime.embeddings([q['question'] for q in queries],allow_network=ALLOW_NETWORK)
qvectors={q['query_id']:v for q,v in zip(queries,question_vectors,strict=True)}
for pf in profiles:runtime.profile(pf)
# 검색 결과만 입력 해시로 재사용합니다. 새 답변은 매 회차·방식마다 별도 유료 identity입니다.
search_cache={};all_plans=[];retrievals=[];answers=[];pending_review=None;stopped={}
for rep in range(1,4):
 active=profiles
 if rep>1:
  gatepath=PATH/'round1_review_gate.json'
  if not gatepath.exists():pending_review=1;break
  gate=P.F.read_json(gatepath)
  assert gate['execution_manifest_sha256']==P.sha(manifest)
  round1=[a for a in answers if a['repeat']==1]
  assert gate['round1_answers_sha256']==P.sha(round1)
  assert gate['source_review_sha256']==P.sha(P.F.read_json(PATH/'round1_source_review.json'))
  active=[pf for pf in profiles if gate['continue_repeats'][pf]]
  stopped={pf:gate['stop_reasons'][pf] for pf in profiles if pf not in active}
  if not active:break
 round_plans={};routes={};guides={}
 for q in queries:
  identity=f"{q['query_id']}:parser:r{rep}";api=ContractCache(49,'parser',identity,allow_network=ALLOW_NETWORK)
  kwargs=dict(model='gpt-6-luna',instructions=PROMPT,input=[dict(role='user',content=q['question'])],text_format=P.E.QueryPlan,tools=[],store=False,max_output_tokens=3000)
  key=api.key(kwargs);path=PATH/'plans'/(identity+'.json')
  if path.exists():row=P.F.read_json(path)
  else:
   try:
    response=api.parse(**kwargs);value=response.output_parsed.model_dump(mode='json') if response.output_parsed else None
    status='ok' if value else 'failed';error=None
   except Exception as exc:
    if not api.events:raise
    value=None;status='failed';error=f'{type(exc).__name__}: {exc}'
   row=dict(identity=identity,query_id=q['query_id'],repeat=rep,question=q['question'],question_sha256=P.sha(q['question']),
    manifest_sha256=P.sha(manifest),request_key=key,status=status,error=error,plan=value,events=api.events)
   write('plans/'+identity+'.json',row)
  P.validate_parser_row(row,identity=identity,query_id=q['query_id'],repeat=rep,question=q['question'],manifest=manifest,request_key=key,cache=api.cache)
  round_plans[q['query_id']]=row;all_plans.append(row)
  route=selected_route(q['question'],row['plan'],row['status']!='ok');routes[q['query_id']]=route
  for pf in active:
   guides[q['query_id']+':'+pf]=guidance(route,pf) if selected['selected_by_profile'][pf]=='B' else dict(text=None,used=[],skipped=[],reason='selected_A')
  print('parser',identity,status if not path.exists() else row['status'],flush=True)
 auxtexts=sorted({v['text'] for v in guides.values() if v['text']})
 contractpath=PATH/'embedding_text_contract.json'
 previous_texts=P.F.read_json(contractpath)['text_sha256'] if contractpath.exists() else []
 write('embedding_text_contract.json',dict(execution_manifest_sha256=P.sha(manifest),text_sha256=sorted(set(previous_texts)|{P.sha(t) for t in auxtexts})))
 frozen(f'round{rep}_guidance.json',dict(records=guides,parser_rows_sha256=P.sha(round_plans),no_gold_labels=True))
 auxvectors=dict(zip(auxtexts,runtime.embeddings(auxtexts,allow_network=ALLOW_NETWORK),strict=True)) if auxtexts else {}
 round_retrievals=[]
 for q in queries:
  trusted=P.explicit_scope(q['question'],catalog);route=routes[q['query_id']]
  for pf in active:
   # 같은 회차에서 방식 순서를 번갈아 배치하고 답변은 모두 신규 ID로 생성합니다.
   order=modes if (rep+int(q['query_id'][1:]))%2==0 else list(reversed(modes))
   for mode in order:
    identity=f"{q['query_id']}:{pf}:{mode}:r{rep}";g=guides[q['query_id']+':'+pf]
    scope=trusted if mode=='U' else route['scope'];aux=g['text'] if mode=='S' else None
    binding=P.sha(dict(question=q['question'],profile=pf,mode=mode,repeat=rep,route=route if mode=='S' else None,
     scope=scope,guidance=g if mode=='S' else None,index=runtime.index_identity[pf],manifest=P.sha(manifest)))
    path=PATH/'retrieval'/(identity+'.json')
    searchkey=P.sha(dict(question=q['question'],profile=pf,scope=scope,auxiliary=aux))
    if path.exists():
     r=P.F.read_json(path);assert r['input_binding']==binding and r['identity']==identity
    else:
     start=time.perf_counter();reuse=None
     if mode=='S' and route['route']=='clarify':result=dict(cards=[],evidence=[],trace={});reuse='clarification_boundary'
     elif searchkey in search_cache:result=copy.deepcopy(search_cache[searchkey]);reuse='same_search_inputs'
     else:
      supplement=dict(question=aux,vector=auxvectors[aux],scope=scope) if aux else None
      result=runtime.search(q['question'],qvectors[q['query_id']],pf,scope,supplement=supplement)
     r=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,repeat=rep,status='ok',result=result,
      route=route if mode=='S' else dict(route='unstructured',scope=trusted),input_binding=binding,reuse=reuse,wall_seconds=time.perf_counter()-start)
     write('retrieval/'+identity+'.json',r)
    search_cache[searchkey]=r['result'];round_retrievals.append(r);retrievals.append(r)
    print('retrieval',identity,r['reuse'] or 'fresh',flush=True)
 payloads={r['identity']:P.sha([dict(role='user',content=serialize_answer_payload(next(q['question'] for q in queries if q['query_id']==r['query_id']),r['result']['evidence']))]) for r in round_retrievals}
 contractfile=PATH/'answer_payload_contract.json'
 contract=P.F.read_json(contractfile) if contractfile.exists() else dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity={},schema_sha256_by_identity={})
 assert contract['execution_manifest_sha256']==P.sha(manifest)
 for r in round_retrievals:
  identity=r['identity'];schemahash=answer_schema_hash(r['result']['evidence'])
  if identity in contract['input_sha256_by_identity']:assert contract['input_sha256_by_identity'][identity]==payloads[identity] and contract['schema_sha256_by_identity'][identity]==schemahash
  contract['input_sha256_by_identity'][identity]=payloads[identity];contract['schema_sha256_by_identity'][identity]=schemahash
 write('answer_payload_contract.json',contract)
 # 각 회차 계약은 누적 계약과 달리 변경되지 않으며 이전 checkpoint의 해시를 보존합니다.
 frozen(f'round{rep}_answer_payload_contract.json',prefix_contract(contract,rep))
 for r in round_retrievals:
  identity=r['identity'];q=next(q for q in queries if q['query_id']==r['query_id']);path=PATH/'answers'/(identity+'.json')
  bound_contract=P.F.read_json(PATH/f'round{rep}_answer_payload_contract.json')
  if path.exists():
   a=P.F.read_json(path);assert a['retrieval_sha256']==P.sha(r) and a['repeat']==rep
  else:
   start=time.perf_counter();events=[];error=None
   if not r['result']['evidence']:
    text='발급사와 정확한 스카이패스 카드명을 알려주세요.' if r['route']['route']=='clarify' else '검색된 근거가 없습니다.'
    answer=dict(answer_status='insufficient_evidence',answer_text=text,recommendations=[],claims=[]);usage={}
   else:
    api=ContractCache(49,'answer',identity,allow_network=ALLOW_NETWORK)
    try:
     parsed,usage=OpenAIService(api_key=None,client=api,llm_model='gpt-6-luna',answer_payload_bytes=64000).answer(q['question'],r['result']['evidence']);answer=parsed.model_dump(mode='json')
    except Exception as exc:
     if not api.events:raise
     answer=None;usage={};error=f'{type(exc).__name__}: {exc}'
    events=api.events
   a=dict(identity=identity,query_id=q['query_id'],profile=r['profile'],mode=r['mode'],repeat=rep,status='ok' if answer is not None else 'failed',
    answer=answer,usage=usage,events=events,error=error,reuse=None,retrieval_sha256=P.sha(r),input_sha256=payloads[identity],
    wall_seconds=time.perf_counter()-start,manifest_sha256=P.sha(manifest),payload_contract_sha256=P.sha(bound_contract),request_keys=[e['key'] for e in events])
   write('answers/'+identity+'.json',a)
  R.validate_generated_answer(a,q['question'],r['result']['evidence'],stage=49,identity=identity,manifest=manifest,contract=bound_contract)
  answers.append(a)
  write('progress.json',dict(phase='answer',repeat=rep,completed=len(answers),expected_maximum=384,last_identity=identity,ledger=P.ledger()))
  print('answer',identity,a['status'],flush=True)
 resource_row=dict(resources=runtime.resources(),actual_answer_conditions=sum(a['repeat']==rep for a in answers))
 if not (PATH/f'round{rep}_resources.json').exists():write(f'round{rep}_resources.json',resource_row)
 else:write(f'round{rep}_latest_replay_resources.json',resource_row)
write('execution_results.json',dict(parser_rows=all_plans,retrievals=retrievals,answers=answers,pending_review=pending_review,stopped_profiles=stopped,resources=runtime.resources()))
print('완료조건',len(answers),'검토대기',pending_review,'프로필중단',stopped,'누적비용',P.ledger())

/home/sms/miniconda3/envs/skn25/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


parser H01:parser:r1 ok


parser H02:parser:r1 ok


parser H03:parser:r1 ok


parser H04:parser:r1 ok


...

answer H32:card_page_section_benefit:S:r1 ok


answer H32:card_page_section_benefit:U:r1 ok


answer H32:parent_child_bundle:S:r1 ok


answer H32:parent_child_bundle:U:r1 ok


완료조건 128 검토대기 None 프로필중단 {'card_page_section_benefit': 'H15: 같은 근거가 있지만 후보 답변 전체에서 공과금 할인 일1회 조건 누락. 기준은 보존. 사전 새 중요조건 손실0 기준 미충족.', 'parent_child_bundle': 'H18: 같은 근거가 있지만 후보의 롯데마트&MAXX 설명에서 발급첫달 생활업종 할인제외 누락. 기준은 보존. 사전 새 중요조건 손실0 기준 미충족.'} 누적비용 {'record_count': 319, 'known_priced_records': 319, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.234905430', 'tokens': {'input_tokens': 1746667, 'output_tokens': 251030, 'ordinary_input_tokens': 948, 'cached_input_tokens': 948759, 'cache_write_tokens': 796960, 'embedding_tokens': 9402}, 'billed_cost_usd': None}


In [4]:
# 방식명을 가린 출력과 실제 소유 인용 근거를 검토합니다. 정답은 생성 입력에 전달하지 않았습니다.
labels={q['query_id']:q for q in annotations['queries']};lookup={r['identity']:r for r in retrievals};assignments=[]
for rep in sorted({a['repeat'] for a in answers}):
 for q in queries:
  for pf in profiles:
   pair=[a for a in answers if a['repeat']==rep and a['query_id']==q['query_id'] and a['profile']==pf]
   if len(pair)!=2:continue
   pair=sorted(pair,key=lambda a:P.sha('mode-hidden-review:'+a['identity']))
   pid=P.sha(dict(query=q['query_id'],profile=pf,repeat=rep))[:16]
   sides={}
   for side,a in zip(['left','right'],pair,strict=True):
    ev=lookup[a['identity']]['result']['evidence'];cited={cid for cl in (a['answer'] or {}).get('claims',[]) for cid in cl['citations']}
    sides[side]=dict(status=a['status'],answer=a['answer'],evidence=ev)
   frozen('review_packets/'+pid+'.json',dict(packet_id=pid,question=q['question'],query_id=q['query_id'],profile=pf,repeat=rep,
    requirements=labels[q['query_id']]['required_meanings'],non_exhaustive_gold=labels[q['query_id']]['gold_cards'],**sides))
   assignments.append(dict(packet_id=pid,query_id=q['query_id'],profile=pf,repeat=rep,left=pair[0]['identity'],right=pair[1]['identity']))
write('review_assignments.json',dict(records=assignments,source_answer_rows_sha256=P.sha(answers)))
# 미실행을 실패나 동률로 바꾸지 않습니다.
executed={a['identity'] for a in answers}
write('condition_status.json',dict(records=[dict(identity=identity,status='executed' if identity in executed else 'not_executed',
 reason=None if identity in executed else 'round1_source_review_pending' if pending_review else 'early_stop_of_profile') for identity in manifest['allowed_identities']['answer']]))
changed=[];allowed=[]
allowance=P.F.read_json(PATH/'authorized_metric_append.json') if (PATH/'authorized_metric_append.json').exists() else None
for record in P.F.read_json(PATH/'protected_files.json')['files']:
 source=ROOT/record['path']
 if not source.exists():changed.append(record['path']);continue
 if P.F.file_record(source)!=record:
  body=source.read_bytes()
  if allowance and record['path']==allowance['path'] and P.sha(body)==allowance['current_sha256'] and P.sha(body[:record['size']])==record['sha256']:allowed.append(record['path'])
  else:changed.append(record['path'])
write('preservation_verification.json',dict(passed=not changed,changed=changed,allowed_document_appends=allowed))
assert not changed,changed
print('방식명 숨긴 검토 묶음',len(assignments),'원본 보존 통과. 최종 품질 판정은 검토 후 기록합니다.')

방식명 숨긴 검토 묶음 64 원본 보존 통과. 최종 품질 판정은 검토 후 기록합니다.
